<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
<br>汉化的库: <a href="https://github.com/GoatCsu/CN-LLMs-from-scratch.git">https://github.com/GoatCsu/CN-LLMs-from-scratch.git</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第四章 课后练习

In [32]:
from importlib.metadata import version
import torch
print("torch version:", version("torch"))

torch version: 2.11.0+cu128


# 练习 4.1：前馈模块与注意力模块中的参数

In [33]:
from gpt import TransformerBlock
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}
block = TransformerBlock(GPT_CONFIG_124M)
print(block)  # 输出模块的层级结构
'''
TransformerBlock(
  (norm1): 层归一化（注意力层前）
  (att): 多头注意力模块(
    (W_q): 线性层（查矩阵，输入特征数=768，输出特征数=768，无偏置）
    (W_k): 线性层（键矩阵，输入特征数=768，输出特征数=768，无偏置）
    (W_v): 线性层（值矩阵，输入特征数=768，输出特征数=768，无偏置）
    (out_proj): 线性层（输出投影，输入特征数=768，输出特征数=768，有偏置）
    (dropout): Dropout层（丢弃率p=0.1，非原地操作）
  )
  (drop_shortcut): Dropout层（残差连接用，丢弃率p=0.1，非原地操作）
  (norm2): 层归一化（前馈网络前）
  (ff): 前馈网络模块(
    (layers): 序列模块(
      (0): 线性层（输入特征数=768，输出特征数=3072，有偏置）
      (1): GELU激活函数
      (2): 线性层（输入特征数=3072，输出特征数=768，有偏置）
    )
  )
)
'''

TransformerBlock(
  (att): MultiHeadAttention(
    (W_query): Linear(in_features=768, out_features=768, bias=False)
    (W_key): Linear(in_features=768, out_features=768, bias=False)
    (W_value): Linear(in_features=768, out_features=768, bias=False)
    (out_proj): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (ff): FeedForward(
    (layers): Sequential(
      (0): Linear(in_features=768, out_features=3072, bias=True)
      (1): GELU()
      (2): Linear(in_features=3072, out_features=768, bias=True)
    )
  )
  (norm1): LayerNorm()
  (norm2): LayerNorm()
  (drop_shortcut): Dropout(p=0.1, inplace=False)
)


'\nTransformerBlock(\n  (norm1): 层归一化（注意力层前）\n  (att): 多头注意力模块(\n    (W_q): 线性层（查矩阵，输入特征数=768，输出特征数=768，无偏置）\n    (W_k): 线性层（键矩阵，输入特征数=768，输出特征数=768，无偏置）\n    (W_v): 线性层（值矩阵，输入特征数=768，输出特征数=768，无偏置）\n    (out_proj): 线性层（输出投影，输入特征数=768，输出特征数=768，有偏置）\n    (dropout): Dropout层（丢弃率p=0.1，非原地操作）\n  )\n  (drop_shortcut): Dropout层（残差连接用，丢弃率p=0.1，非原地操作）\n  (norm2): 层归一化（前馈网络前）\n  (ff): 前馈网络模块(\n    (layers): 序列模块(\n      (0): 线性层（输入特征数=768，输出特征数=3072，有偏置）\n      (1): GELU激活函数\n      (2): 线性层（输入特征数=3072，输出特征数=768，有偏置）\n    )\n  )\n)\n'

In [34]:
total_params = sum(p.numel() for p in block.ff.parameters())
print(f"前馈网络模块的总参数量: {total_params:,}")
total_params = sum(p.numel() for p in block.att.parameters())
print(f"注意力模块的总参数量: {total_params:,}")

前馈网络模块的总参数量: 4,722,432
注意力模块的总参数量: 2,360,064


- 上述结果是针对单个transformer块的
- 注意力模块的参数量通常约为前馈网络的一半
- 可选地，将结果乘以 12，以计算 124M GPT 模型中所有transformer的参数

**额外内容：数据分析**

- 对于那些有兴趣了解这些参数数量如何从数据上计算的人，下面是详细的分析（假设 `emb_dim=768`）：

前馈模块：

- 第一个 `Linear` 层：768 输入 × 4×768 输出 + 4×768 偏置单元 = 2,362,368
- 第二个 `Linear` 层：4×768 输入 × 768 输出 + 768 偏置单元 = 2,360,064
- 总计：第一个 `Linear` 层 + 第二个 `Linear` 层 = 2,362,368 + 2,360,064 = 4,722,432

注意力模块：

- `W_query`：768 输入 × 768 输出 = 589,824 
- `W_key`：768 输入 × 768 输出 = 589,824
- `W_value`：768 输入 × 768 输出 = 589,824 
- `out_proj`：768 输入 × 768 输出 + 768 偏置单元 = 590,592
- 总计：`W_query` + `W_key` + `W_value` + `out_proj` = 3×589,824 + 590,592 = 2,360,064

# 练习4.2 初始化GPT模块

- **GPT2-small**（已实现的 124M 配置）：
    - "emb_dim" = 768
    - "n_layers" = 12
    - "n_heads" = 12

- **GPT2-medium**：
    - "emb_dim" = 1024
    - "n_layers" = 24
    - "n_heads" = 16

- **GPT2-large**：
    - "emb_dim" = 1280
    - "n_layers" = 36
    - "n_heads" = 20

- **GPT2-XL**：
    - "emb_dim" = 1600
    - "n_layers" = 48
    - "n_heads" = 25

In [35]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

def get_config(base_config, model_name="gpt2-small"):
    GPT_CONFIG = base_config.copy()  # 避免修改原配置
    if model_name == "gpt2-small":
        GPT_CONFIG["emb_dim"] = 768
        GPT_CONFIG["n_layers"] = 12
        GPT_CONFIG["n_heads"] = 12
    elif model_name == "gpt2-medium":
        GPT_CONFIG["emb_dim"] = 1024
        GPT_CONFIG["n_layers"] = 24
        GPT_CONFIG["n_heads"] = 16
    elif model_name == "gpt2-large":
        GPT_CONFIG["emb_dim"] = 1280
        GPT_CONFIG["n_layers"] = 36
        GPT_CONFIG["n_heads"] = 20
    elif model_name == "gpt2-xl":
        GPT_CONFIG["emb_dim"] = 1600
        GPT_CONFIG["n_layers"] = 48
        GPT_CONFIG["n_heads"] = 25
    else:
        raise ValueError(f"Incorrect model name {model_name}")  # 对非法 model_name，抛出 ValueError 异常
    return GPT_CONFIG


def calculate_size(model, device="cpu"):
    model = model.to(device)
    # 计算总参数量
    total_params = sum(p.numel() for p in model.parameters())
    print(f"总参数量: {total_params:,}")
    # 计算考虑权重共享后的有效参数量（减去输出头）
    total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
    print(f"考虑权重共享后的有效参数量: {total_params_gpt2:,}")
    # 计算模型在float32精度下的内存占用
    total_size_bytes = total_params * 4
    total_size_mb = total_size_bytes / (1024 * 1024)
    print(f"float32 精度下的模型内存占用: {total_size_mb:.2f} MB")

In [36]:
from gpt import GPTModel

for model_abbrev in ("small", "medium", "large", "xl"):  # 遍历 GPT-2 的所有 4 个规模
    model_name = f"gpt2-{model_abbrev}"
    CONFIG = get_config(GPT_CONFIG_124M, model_name=model_name)  # 调用 get_config 获取对应配置
    model = GPTModel(CONFIG)
    print(f"\n\n{model_name}:")
    calculate_size(model)



gpt2-small:
总参数量: 163,009,536
考虑权重共享后的有效参数量: 124,412,160
float32 精度下的模型内存占用: 621.83 MB


gpt2-medium:
总参数量: 406,212,608
考虑权重共享后的有效参数量: 354,749,440
float32 精度下的模型内存占用: 1549.58 MB


gpt2-large:
总参数量: 838,220,800
考虑权重共享后的有效参数量: 773,891,840
float32 精度下的模型内存占用: 3197.56 MB


gpt2-xl:
总参数量: 1,637,792,000
考虑权重共享后的有效参数量: 1,557,380,800
float32 精度下的模型内存占用: 6247.68 MB


# 练习4.3 Droppout模块

In [37]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_emb": 0.1,        # NEW: dropout for embedding layers
    "drop_rate_attn": 0.1,       # NEW: dropout for multi-head attention  
    "drop_rate_shortcut": 0.1,   # NEW: dropout for shortcut connections  
    "qkv_bias": False
}

In [38]:
import torch.nn as nn
from gpt import MultiHeadAttention, LayerNorm, FeedForward


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate_attn"], # 新增：多头注意力的 dropout
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

    def forward(self, x):
        # 注意力块的快捷连接
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # 形状 [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # 将原始输入加回来

        # 前馈块的快捷连接
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 将原始输入加回来

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"]) # 新增：embedding 层的 dropout

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # 形状 [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [39]:
import torch

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)